# Copernicus Sentinel-2 thinking without large downloads

Sentinel-2 analysis in full form often requires raster libraries that are not practical in Pyodide. This notebook teaches the concept using web map tile overlays, local vectors, and a small table-driven index exercise.

For production work, move the same concepts to xarray/rioxarray/rasterio in a server or desktop environment.

**Reflection questions:** Why are cloud masks essential for satellite analysis? How does spatial resolution change the kinds of decisions you can support? What metadata belongs beside every raster layer?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
# A tiny spectral-index exercise: calculate NDVI from sample red/NIR reflectance values.
samples = pd.DataFrame([
    {'landcover':'dense vegetation','red':0.05,'nir':0.52},
    {'landcover':'urban surface','red':0.20,'nir':0.24},
    {'landcover':'bare soil','red':0.28,'nir':0.34},
    {'landcover':'water','red':0.03,'nir':0.02},
])
samples['ndvi'] = (samples.nir - samples.red) / (samples.nir + samples.red)
samples

In [ ]:
districts = load_json('montreal_districts.geojson')
m = folium.Map(location=[45.52, -73.60], zoom_start=10, tiles='CartoDB positron')
# Sentinel Hub public demo endpoint can require service changes; keep as optional tile overlay for live web sessions.
folium.GeoJson(districts, name='Analysis boundary', style_function=lambda f: {'fillOpacity':0.08, 'weight':1}).add_to(m)
# Use circle markers as a conceptual spectral sample layer.
for i, row in samples.iterrows():
    folium.CircleMarker([45.49 + i*0.018, -73.64 + i*0.035], radius=12, fill=True, popup=f"{row.landcover}: NDVI {row.ndvi:.2f}").add_to(m)
add_standard_controls(m)
m